# statement_processing_v2.ipynb — 非泄漏、稳健版流水特征
统一窗口（默认 180 天）、仅用训练分位数裁剪（1%~99%）、对重尾 `log1p`，并对缺失 id 兜底，避免泄漏。

In [1]:

# 配置
TRAIN_RAW_STM = "train_bank_statement.csv"
TESTAA_RAW_STM = "testaa_bank_statement.csv"
TESTAB_RAW_STM = "testab_bank_statement.csv"

TRAIN_IDS = "train.csv"
TESTAA_IDS = "testaa.csv"
TESTAB_IDS = "testab.csv"

OUT_TRAIN_V2 = "train_statement_feature_v2.csv"
OUT_TESTAA_V2 = "testaa_statement_feature_v2.csv"
OUT_TESTAB_V2 = "testab_statement_feature_v2.csv"

# 同步默认名（若不存在则写入，不覆盖旧文件）
OUT_TRAIN_DEFAULT = "train_statement_feature.csv"
OUT_TESTAA_DEFAULT = "testaa_statement_feature.csv"
OUT_TESTAB_DEFAULT = "testab_statement_feature.csv"

WINDOW_DAYS = 180
CLIP_LOW_PCT = 0.01
CLIP_HIGH_PCT = 0.99
APPLY_LOG1P = True
QUANTILE_JSON = "stm_v2_clip_quantiles.json"


In [2]:

import os, json, numpy as np, pandas as pd

def _read_ids(path):
    if not os.path.exists(path): return None
    try:
        df = pd.read_csv(path, usecols=['id'])
    except Exception:
        df = pd.read_csv(path)
        if 'id' not in df.columns:
            raise ValueError(f"{path} 缺少 'id' 列")
        df = df[['id']]
    return df['id'].drop_duplicates().reset_index(drop=True)

def _safe_to_datetime_from_unix(s):
    return pd.to_datetime(s, unit='s', utc=True, errors='coerce').dt.tz_convert(None)

def _aggregate_fixed_window(df_id, now_dt, win_days=180):
    start_dt = now_dt - pd.Timedelta(days=win_days)
    part = df_id[df_id['time'].between(start_dt, now_dt)]
    if len(part)==0:
        return dict(
            total_income=0.0, total_expense=0.0, net_income=0.0,
            income_count=0, expense_count=0, tx_count=0,
            active_days=0, avg_income=0.0, avg_expense=0.0,
            income_std=0.0, expense_std=0.0, p95_income=0.0, p95_expense=0.0,
            last_tx_days=win_days+1, first_tx_days=win_days+1,
            income_expense_ratio=0.0
        )
    part = part.copy()
    part['date'] = _safe_to_datetime_from_unix(part['time']).dt.date if 'date' not in part else part['date']
    if 'time' in part and not np.issubdtype(part['time'].dtype, np.datetime64):
        part['time'] = _safe_to_datetime_from_unix(part['time'])

    income_mask  = part['direction'].astype(int) == 0
    expense_mask = part['direction'].astype(int) == 1

    income_amt  = pd.to_numeric(part.loc[income_mask, 'amount'], errors='coerce')
    expense_amt = pd.to_numeric(part.loc[expense_mask, 'amount'], errors='coerce')

    total_income   = float(np.nansum(income_amt))
    total_expense  = float(np.nansum(expense_amt))
    net_income     = total_income - total_expense
    income_count   = int(np.sum(~np.isnan(income_amt)))
    expense_count  = int(np.sum(~np.isnan(expense_amt)))
    tx_count       = int(len(part))
    active_days    = int(pd.Series(part['date']).nunique())

    avg_income   = float(np.nanmean(income_amt)) if income_count>0 else 0.0
    avg_expense  = float(np.nanmean(expense_amt)) if expense_count>0 else 0.0
    income_std   = float(np.nanstd(income_amt)) if income_count>1 else 0.0
    expense_std  = float(np.nanstd(expense_amt)) if expense_count>1 else 0.0
    p95_income   = float(np.nanquantile(income_amt, 0.95)) if income_count>0 else 0.0
    p95_expense  = float(np.nanquantile(expense_amt, 0.95)) if expense_count>0 else 0.0

    last_tx_days  = int((now_dt - part['time'].max()).days)
    first_tx_days = int((now_dt - part['time'].min()).days)

    ratio = float(total_income / total_expense) if total_expense not in (0, 0.0) else float(total_income)

    return dict(
        total_income=total_income, total_expense=total_expense, net_income=net_income,
        income_count=income_count, expense_count=expense_count, tx_count=tx_count,
        active_days=active_days, avg_income=avg_income, avg_expense=avg_expense,
        income_std=income_std, expense_std=expense_std,
        p95_income=p95_income, p95_expense=p95_expense,
        last_tx_days=last_tx_days, first_tx_days=first_tx_days,
        income_expense_ratio=ratio
    )

def build_features_from_raw(path_csv, ids_hint=None, window_days=180):
    usecols = ['id','time','direction','amount']
    df = pd.read_csv(path_csv, usecols=usecols)
    df['time'] = _safe_to_datetime_from_unix(df['time'])
    df = df.dropna(subset=['id','time','direction','amount']).copy()
    df = df.sort_values(['id','time'])

    now_by_id = df.groupby('id')['time'].max()
    feats = []
    for gid, part in df.groupby('id', sort=False):
        now_dt = now_by_id.loc[gid]
        agg = _aggregate_fixed_window(part, now_dt, win_days=window_days)
        agg['id'] = gid
        feats.append(agg)
    feat_df = pd.DataFrame(feats)
    if ids_hint is not None:
        ids = pd.DataFrame({'id': ids_hint})
        feat_df = ids.merge(feat_df, on='id', how='left')
    stm_cols = [c for c in feat_df.columns if c!='id']
    feat_df['has_stm'] = (feat_df[stm_cols].notna().any(axis=1)).astype(int)
    feat_df[stm_cols] = feat_df[stm_cols].fillna(0)
    return feat_df

def robust_clip_and_log(train_df, df, low=0.01, high=0.99, apply_log1p=True):
    q = {}
    out = df.copy()
    num_cols = [c for c in df.columns if c!='id' and pd.api.types.is_numeric_dtype(df[c])]
    for c in num_cols:
        lo = float(np.nanquantile(train_df[c].values, low))
        hi = float(np.nanquantile(train_df[c].values, high))
        if not np.isfinite(lo): lo = float(np.nanmin(train_df[c].values))
        if not np.isfinite(hi): hi = float(np.nanmax(train_df[c].values))
        if lo == hi:
            lo, hi = float(np.nanmin(train_df[c].values)), float(np.nanmax(train_df[c].values))
        q[c] = dict(lo=lo, hi=hi)
        out[c] = out[c].clip(lower=lo, upper=hi)
        if apply_log1p and any(k in c for k in ['total','avg','median','count','std','ratio','income','expense','tx']):
            out[c] = np.log1p(out[c])
    return out, q

def fallback_from_existing_feature(path_feature_csv, ids_hint=None, train_ref=None, low=0.01, high=0.99, apply_log1p=True):
    if not os.path.exists(path_feature_csv):
        return None, {}
    df = pd.read_csv(path_feature_csv)
    if 'id' not in df.columns:
        raise ValueError(f"{path_feature_csv} 缺少 'id' 列")
    if ids_hint is not None:
        ids = pd.DataFrame({'id': ids_hint})
        df = ids.merge(df, on='id', how='left').fillna(0)
    ref = df.copy() if train_ref is None else train_ref
    out, q = robust_clip_and_log(ref, df, low=low, high=high, apply_log1p=apply_log1p)
    stm_cols = [c for c in out.columns if c!='id']
    out['has_stm'] = (out[stm_cols].notna().any(axis=1)).astype(int)
    out[stm_cols] = out[stm_cols].fillna(0)
    return out, q

def write_if_absent(path_dst, df):
    if df is None: return
    if not os.path.exists(path_dst):
        df.to_csv(path_dst, index=False, encoding='utf-8')
        print("[SAVE]", path_dst, "(new)")
    else:
        print("[SKIP] 已存在，不覆盖：", path_dst)


In [3]:

# 主流程
train_ids = _read_ids(TRAIN_IDS)
testaa_ids = _read_ids(TESTAA_IDS)
testab_ids = _read_ids(TESTAB_IDS)

# 训练
if os.path.exists(TRAIN_RAW_STM):
    tr_v2_raw = build_features_from_raw(TRAIN_RAW_STM, ids_hint=train_ids, window_days=WINDOW_DAYS)
    tr_v2 = tr_v2_raw.copy()
else:
    tr_v2, _ = fallback_from_existing_feature("train_statement_feature.csv", ids_hint=train_ids, train_ref=None,
                                              low=CLIP_LOW_PCT, high=CLIP_HIGH_PCT, apply_log1p=APPLY_LOG1P)

# 分位阈值：仅用训练
tr_v2_for_q = tr_v2.copy()

# testaa
teaa_v2 = None
if testaa_ids is not None:
    if os.path.exists(TESTAA_RAW_STM):
        teaa_raw = build_features_from_raw(TESTAA_RAW_STM, ids_hint=testaa_ids, window_days=WINDOW_DAYS)
    else:
        teaa_raw, _ = fallback_from_existing_feature("testaa_statement_feature.csv", ids_hint=testaa_ids, train_ref=None,
                                                     low=CLIP_LOW_PCT, high=CLIP_HIGH_PCT, apply_log1p=False)
    teaa_v2, _ = robust_clip_and_log(tr_v2_for_q, teaa_raw, low=CLIP_LOW_PCT, high=CLIP_HIGH_PCT, apply_log1p=APPLY_LOG1P)

# testab
teab_v2 = None
if testab_ids is not None:
    if os.path.exists(TESTAB_RAW_STM):
        teab_raw = build_features_from_raw(TESTAB_RAW_STM, ids_hint=testab_ids, window_days=WINDOW_DAYS)
    else:
        teab_raw, _ = fallback_from_existing_feature("testab_statement_feature.csv", ids_hint=testab_ids, train_ref=None,
                                                     low=CLIP_LOW_PCT, high=CLIP_HIGH_PCT, apply_log1p=False)
    teab_v2, _ = robust_clip_and_log(tr_v2_for_q, teab_raw, low=CLIP_LOW_PCT, high=CLIP_HIGH_PCT, apply_log1p=APPLY_LOG1P)

# 训练也裁剪+log1p
tr_v2_final, _ = robust_clip_and_log(tr_v2_for_q, tr_v2, low=CLIP_LOW_PCT, high=CLIP_HIGH_PCT, apply_log1p=APPLY_LOG1P)

# 保存阈值配置（用于审计复现）
with open(QUANTILE_JSON, "w", encoding="utf-8") as f:
    json.dump({'low':CLIP_LOW_PCT,'high':CLIP_HIGH_PCT,'window_days':WINDOW_DAYS}, f, ensure_ascii=False, indent=2)
print("[SAVE]", QUANTILE_JSON)

# 写出
if tr_v2_final is not None:
    tr_v2_final.to_csv(OUT_TRAIN_V2, index=False, encoding='utf-8'); print("[SAVE]", OUT_TRAIN_V2)
if teaa_v2 is not None:
    teaa_v2.to_csv(OUT_TESTAA_V2, index=False, encoding='utf-8'); print("[SAVE]", OUT_TESTAA_V2)
if teab_v2 is not None:
    teab_v2.to_csv(OUT_TESTAB_V2, index=False, encoding='utf-8'); print("[SAVE]", OUT_TESTAB_V2)

# 同步默认名（若不存在）
write_if_absent(OUT_TRAIN_DEFAULT, tr_v2_final)
write_if_absent(OUT_TESTAA_DEFAULT, teaa_v2)
write_if_absent(OUT_TESTAB_DEFAULT, teab_v2)


d:\anaconda\envs\LoanCom37\lib\site-packages\pandas\core\arraylike.py:364: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
d:\anaconda\envs\LoanCom37\lib\site-packages\pandas\core\arraylike.py:364: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


[SAVE] stm_v2_clip_quantiles.json
[SAVE] train_statement_feature_v2.csv
[SAVE] testaa_statement_feature_v2.csv
[SAVE] testab_statement_feature_v2.csv
[SAVE] train_statement_feature.csv (new)
[SAVE] testaa_statement_feature.csv (new)
[SAVE] testab_statement_feature.csv (new)
